# Job shop tutorial, Q1–Q3: optimal schedules with Pyomo

Tutorial: *Week 7 - Job Shop Scheduling Tutorial*, Q1–Q3. Q4–Q5 are reading exercises on the OR-Tools site
(Google Colab).

**The problems.** Each job is a fixed sequence of operations; each operation needs a given machine for a given
time, and a machine does one operation at a time. Find start times that minimise the makespan $C_{\max}$, the
time the last job finishes.

Q1 (machines $M_0$–$M_2$):

| Job | Machines in order | Times |
| --- | --- | --- |
| $J_0$ | $M_0$, $M_1$, $M_2$ | 3, 2, 2 |
| $J_1$ | $M_0$, $M_2$, $M_1$ | 4, 3, 1 |
| $J_2$ | $M_1$, $M_0$, $M_2$ | 2, 3, 2 |

Q2 (machines A–D):

| Job | Machines in order | Times |
| --- | --- | --- |
| $J_1$ | A, B, C, D | 3, 2, 4, 3 |
| $J_2$ | B, A, D | 5, 3, 2 |
| $J_3$ | C, D, A, B | 2, 1, 4, 4 |
| $J_4$ | D, C, B | 4, 3, 5 |

Q3 (machines $M_1$–$M_5$):

| Job | Machines in order | Times |
| --- | --- | --- |
| $J_A$ | $M_1$, $M_2$, $M_3$, $M_4$, $M_5$ | 4, 2, 3, 5, 2 |
| $J_B$ | $M_3$, $M_4$, $M_2$, $M_5$ | 2, 1, 3, 4 |
| $J_C$ | $M_1$, $M_2$, $M_3$, $M_4$ | 3, 4, 2, 3 |

In the code jobs and machines are numbered from 0 in every problem (Q2: A = 0, …, D = 3; Q3: $M_1$ = 0, …,
$M_5$ = 4; the first job in each table is Job 0), and "task k" is the job's operation k + 1.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes
about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Optimal makespans Q1 = 12, Q2 = 16, Q3 = 18 (checked by exhaustive search), each printed
with its schedule machine by machine and drawn as a Gantt chart. Several schedules reach each optimum, so your
start times may differ from someone else's; the makespans will match. In the browser HiGHS prints its solver log
before each answer, even though the code does not ask for it; the results are not affected.

Q1 correction: earlier versions of the sheet gave job J2 the machine sequence M1, M3, M2. M3 was a typo (the problem has three machines, M0–M2, and each job visits each machine once), so J2's sequence is M1, M0, M2.

*Original script: `5 Source Files/Python/jss_tutorial_answers_pyomo.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

The model as a function. `jobs` is a list of jobs; each job is a list of (machine, time) pairs in operation
order. The function builds the MILP from the notes: a start time `s` for every operation, one binary `y` for
every pair of operations that share a machine, and the makespan `cmax`. It solves the model with HiGHS, prints
the schedule machine by machine and draws a Gantt chart (one row per machine, one colour per job).

In [ ]:
import matplotlib.pyplot as plt
import pyomo.environ as pyo


def solve_job_shop(jobs, title=""):
    ops = [(j, k) for j, job in enumerate(jobs) for k in range(len(job))]
    mach = {(j, k): jobs[j][k][0] for (j, k) in ops}
    p = {(j, k): jobs[j][k][1] for (j, k) in ops}
    big_m = sum(p.values())  # no schedule is longer than doing everything one after another
    pairs = [(a, b) for a in ops for b in ops if a < b and mach[a] == mach[b]]

    m = pyo.ConcreteModel()
    m.s = pyo.Var(ops, within=pyo.NonNegativeReals)   # start times
    m.y = pyo.Var(pairs, within=pyo.Binary)           # 1 if the first op of the pair goes first
    m.cmax = pyo.Var(within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=m.cmax, sense=pyo.minimize)

    # 1) precedence: each operation starts after the job's previous operation has finished
    m.order = pyo.ConstraintList()
    for (j, k) in ops:
        if k > 0:
            m.order.add(m.s[j, k] >= m.s[j, k - 1] + p[j, k - 1])
    # 2) no overlap: for each pair of operations on the same machine, y = 1 puts the first one of the pair
    #    first, y = 0 the second; big_m switches off the constraint that does not apply
    m.no_overlap = pyo.ConstraintList()
    for (a, b) in pairs:
        m.no_overlap.add(m.s[b] >= m.s[a] + p[a] - big_m * (1 - m.y[a, b]))
        m.no_overlap.add(m.s[a] >= m.s[b] + p[b] - big_m * m.y[a, b])
    # 3) the makespan is at least the finishing time of every job's last operation
    m.makespan = pyo.ConstraintList()
    for j, job in enumerate(jobs):
        last = (j, len(job) - 1)
        m.makespan.add(m.cmax >= m.s[last] + p[last])

    pyo.SolverFactory('appsi_highs').solve(m)

    print(f"{title}  optimal makespan = {pyo.value(m.cmax):g}")
    for (j, k) in sorted(ops, key=lambda o: (mach[o], pyo.value(m.s[o]))):
        start = round(pyo.value(m.s[j, k]), 6) + 0.0  # hide solver round-off such as 8e-14 when printing
        print(f"  Machine {mach[j, k]}: Job {j} task {k}  start {start:g}  end {start + p[j, k]:g}")

    fig, ax = plt.subplots(figsize=(8, 0.6 * (max(mach.values()) + 2)))
    for (j, k) in ops:
        start = pyo.value(m.s[j, k])
        ax.barh(mach[j, k], p[j, k], left=start, color=f"C{j}", edgecolor="black")
        ax.text(start + p[j, k] / 2, mach[j, k], f"J{j}", ha="center", va="center", fontsize=8)
    ax.set_yticks(sorted(set(mach.values())))
    ax.set_yticklabels([f"Machine {i}" for i in sorted(set(mach.values()))])
    ax.invert_yaxis()
    ax.set_xlabel("Time")
    ax.set_title(f"{title}: makespan {pyo.value(m.cmax):g}")
    plt.show()
    return pyo.value(m.cmax)

Q1: three jobs on machines $M_0$–$M_2$. Expected makespan 12.

In [ ]:
# Each job: list of (machine, processing time) in operation order.
# Machines are numbered from 0 (Problems 2 and 3: A/M1 = 0, B/M2 = 1, ...).

# Problem 1  (J0: M0 M1 M2 / J1: M0 M2 M1 / J2: M1 M0 M2)
jobs_1 = [
    [(0, 3), (1, 2), (2, 2)],
    [(0, 4), (2, 3), (1, 1)],
    [(1, 2), (0, 3), (2, 2)]
]
solve_job_shop(jobs_1, "Problem 1")

Q2: four jobs on machines A–D (numbered 0–3 in the code). Expected makespan 16.

In [ ]:
# Problem 2  (machines A, B, C, D = 0, 1, 2, 3)
jobs_2 = [
    [(0, 3), (1, 2), (2, 4), (3, 3)],
    [(1, 5), (0, 3), (3, 2)],
    [(2, 2), (3, 1), (0, 4), (1, 4)],
    [(3, 4), (2, 3), (1, 5)]
]
solve_job_shop(jobs_2, "Problem 2")

Q3: three jobs on five machines $M_1$–$M_5$ (numbered 0–4 in the code). Expected makespan 18.

In [ ]:
# Problem 3  (machines M1..M5 = 0..4)
jobs_3 = [
    [(0, 4), (1, 2), (2, 3), (3, 5), (4, 2)],
    [(2, 2), (3, 1), (1, 3), (4, 4)],
    [(0, 3), (1, 4), (2, 2), (3, 3)]
]
solve_job_shop(jobs_3, "Problem 3")

## What do you notice?

In Q1, machine $M_0$ has $3 + 4 + 3 = 10$ units of work, more than any other machine, and no job needs more than
$4 + 3 + 1 = 8$ units in total. So the makespan cannot be below 10. Why is the optimum 12 and not 10?

<details><summary>Answer</summary>

Look at the operation that $M_0$ processes **last**. $M_0$ cannot finish its 10 units of work before time 10, so
that operation ends at 10 or later. Every operation on $M_0$ is followed by more work for its job: $J_0$ still
needs $2 + 2 = 4$ units, $J_1$ needs $3 + 1 = 4$ and $J_2$ needs $2$. So the last job finishes at $10 + 2 = 12$
at the earliest, and the solver's schedule reaches 12: it is optimal. Bounds like this are how branch and bound
proves a schedule optimal without trying every order.
</details>
